In [0]:
%run ../setup/config

In [0]:
%run ../setup/utils

In [0]:
df = spark.read.format("delta").load(f"{bronze_folder_path}/movies_metadata")
df.printSchema()
display(df)

In [0]:
collection_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("poster_path", StringType(), True),
    StructField("backdrop_path", StringType(), True)
])

df_parsed = flatten_nested_column_silver(df, "collection", "belongs_to_collection", collection_schema)
display(df_parsed)


In [0]:
df_silver = df_parsed \
  .filter(
    (df_parsed.runtime.isNotNull())
    & (df_parsed.vote_average.isNotNull())
    & (df_parsed.vote_count.isNotNull())
  ) \
  .select(
    "id",
    "title",
    "genres", 
    "vote_average",
    "vote_count",
    "budget", 
    "original_language", 
    "overview", 
    "popularity",
    "release_date",
    "revenue",
    "runtime",
    "ingest_timestamp",
    "collection_id",
    "collection_name",
  ).sort("vote_average", ascending=False)
display(df_silver)

In [0]:
df_silver.write.mode("overwrite").format("delta").save(f"{silver_folder_path}/movies_metadata")

In [0]:
genre_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
])

movie_genres_df = explode_nested_json_array_silver(
    df_parsed,   # the df that still has the genres string
    "genres",
    "id",                 # movie id, kept as the key
    genre_schema,
    ["id", "name"],
)

display(movie_genres_df)


In [0]:
movie_genres_df.write.format("delta").save(f"{silver_folder_path}/movie_genres")